# WAF Gen 3 — ทดลองและเทรนบน Google Colab (ฟรี)

Notebook นี้รัน pipeline Gen 3 จาก branch `trainmodelgen3` แล้วเก็บผลไว้ใน Google Drive (`MyDrive/waf_ml/`) ให้วิเคราะห์ต่อ

**ตั้งค่าที่ cell แรก แล้ว Runtime → Run all** (เลือก CPU ไม่ต้องใช้ GPU)

| `EXPERIMENT_MODE` | ทำอะไร | เวลาโดยประมาณ |
| :--- | :--- | :--- |
| `"full"` | `ml/experiment_value_features.py`: 6 config × 3 holdout fold + leave-one-dataset-out | ~1 ชม. |
| `"lofo"` | เฉพาะ leave-one-dataset-out | ~15 นาที |
| `"gate"` | ไม่เทรน — ประเมิน Promotion Gate จากรายงานที่มีใน Drive | ~5 นาที |

`RUN_TRAINER` / `RUN_UNIT_EXPERIMENT` = เทรน hybrid candidate / โมเดลราย unit (1–2 ชม. ต่อขั้น)

รอบแรกของ runtime ใหม่ใช้เวลาเพิ่ม ~10 นาที (ติดตั้ง + โหลด dataset จาก cache ใน Drive; ครั้งแรกสุดดาวน์โหลด public dataset ~1.7 GB)

**Promotion Gate 3.1-G.0** (`WAF_GEN3_ROADMAP.md`, ทุก dataset น้ำหนักเท่ากัน): benign ≥ 98.5% ทุก dataset · attack เฉลี่ยบนเว็บที่รู้จัก ≥ 85% · attack เฉลี่ยบน dataset ที่ไม่เคยเห็น (ที่ benign 98.5%) ≥ 70% · ไม่มี dataset ไหนต่ำกว่า 40% · scenario ปกติ 37/37, โจมตี ≥ 23/26, stress ≥ 60%

**ข้อควรระวัง**
- ถ้าขั้นไหนล้ม notebook จะหยุดพร้อมบอกไฟล์ log — ส่งข้อความ error ให้ Claude แก้ แล้วกด Run all ใหม่ได้ (cache ใน Drive ทำให้ไม่ต้องเริ่มใหม่ทั้งหมด)
- ไม่ต้องกด File → Save ลง GitHub — ผลถูกเก็บลง Drive อัตโนมัติ
- ข้อมูลจาก VPS (`ml/dataset/*.jsonl`, `ml/telemetry/`) เป็น traffic จริงจาก production — ใส่ใน `MyDrive/waf_ml/private/` ก็ต่อเมื่อได้รับอนุญาตแล้วเท่านั้น
- ห้ามแตะ VPS production จาก notebook นี้

In [1]:
# ===== ตั้งค่า =====
REPO = "jakkaret/waf_project"          # GitHub owner/repo
BRANCH = "trainmodelgen3"
USE_DRIVE = True                        # เก็บ dataset/cache/ผลลัพธ์ไว้ใน Google Drive
DRIVE_DIR = "/content/drive/MyDrive/waf_ml"
# ไฟล์ข้อมูล VPS แบบ private (สร้างตาม ml/TRAINING_SETUP.md) — วางไว้ที่ DRIVE_DIR/private/ หรือปล่อยว่างเพื่อใช้ public data อย่างเดียว
PRIVATE_TARBALL_GLOB = DRIVE_DIR + "/private/waf_ml_private_data_*.tar.gz"

# การทดลอง value features:
#   "full" = 3 holdout fold + leave-one-dataset-out (~1 ชม.)   "lofo" = leave-one-dataset-out อย่างเดียว (~15 นาที)
#   "gate" = ไม่เทรนอะไร ประเมิน promotion gate จากรายงานที่มีอยู่แล้วใน Drive (~5 นาที)
EXPERIMENT_MODE = "full"
RUN_TRAINER = False                     # เทรน hybrid candidate ใหม่ (ต้องใช้ก่อน RUN_UNIT_EXPERIMENT)
RUN_UNIT_EXPERIMENT = False
assert EXPERIMENT_MODE in ("full", "lofo", "gate"), EXPERIMENT_MODE

import os, time, glob, subprocess
REPO_DIR = "/content/waf_project"
VENV_PY = "/content/venv/bin/python"
SESSION_START = time.time()
os.environ["PYTHONUNBUFFERED"] = "1"   # log ไหลออกทันที

def run(cmd, log=None, check=True, label=None):  # label: แสดงแทน cmd (เช่นคำสั่งที่มี token)
    """รันคำสั่ง shell: แสดงผลทีละบรรทัด + เขียน log และ *หยุด Run all* ถ้าคำสั่งล้ม (exit code != 0).

    `!cmd | tee` เดิมไม่เคยหยุด: สคริปต์ที่ crash ทำให้ cell สรุปไปแสดงรายงานเก่าแทน
    """
    fp = open(log, "w", encoding="utf-8") if log else None
    # stdin ปิด + GIT_TERMINAL_PROMPT=0: คำสั่งที่ถามคำถาม (uv venv, git ขอรหัสผ่าน) จะล้มทันทีแทนการค้างรอ input
    # pipefail: ถ้าคำสั่งใดใน pipe ล้ม ถือว่าล้มทั้งหมด
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, GIT_TERMINAL_PROMPT="0"))
    for line in p.stdout:
        print(line, end="")
        if fp:
            fp.write(line)
            fp.flush()
    rc = p.wait()
    if fp:
        fp.close()
    if check and rc != 0:
        raise RuntimeError(f"คำสั่งล้ม (exit {rc}): {label or cmd}" + (f" — ดู {log}" if log else ""))
    return rc

print("CPU:", os.cpu_count(), "| RAM GB:", round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1))

CPU: 2 | RAM GB: 13.6


In [2]:
# ===== Google Drive + clone repo =====
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

# repo private: เพิ่ม Colab Secret ชื่อ GITHUB_TOKEN (fine-grained token, สิทธิ์ Contents: read) ที่แถบ 🔑 ด้านซ้าย
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if not os.path.isdir(REPO_DIR + "/.git"):
    run(f"git clone -q --branch {BRANCH} {url} {REPO_DIR}", label=f"git clone {REPO} ({BRANCH})")
else:  # runtime เดิม: ดึงโค้ดล่าสุดทับ
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} "
        f"&& git -C {REPO_DIR} reset -q --hard origin/{BRANCH}", label=f"git fetch/reset {BRANCH}")
run(f"git -C {REPO_DIR} remote set-url origin https://github.com/{REPO}.git")
run(f"git -C {REPO_DIR} log --oneline -3")
os.chdir(REPO_DIR)
os.makedirs("colab_logs", exist_ok=True)

# คัดลอก log ขึ้น Drive ทุก 2 นาที (MyDrive/waf_ml/live_logs/) ให้ Claude ตามดูความคืบหน้าผ่าน Google Drive ได้
import threading, shutil
LIVE_LOG_DIR = f"{DRIVE_DIR}/live_logs"
def _sync_logs():
    while True:
        try:
            os.makedirs(LIVE_LOG_DIR, exist_ok=True)
            for f in glob.glob(f"{REPO_DIR}/colab_logs/*.log"):
                shutil.copy2(f, LIVE_LOG_DIR)
            with open(f"{LIVE_LOG_DIR}/heartbeat.txt", "w") as fp:
                fp.write(time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()) + "\n")
        except Exception:
            pass
        time.sleep(120)
if USE_DRIVE and not globals().get("_log_sync_started"):
    threading.Thread(target=_sync_logs, daemon=True).start()
    _log_sync_started = True

Mounted at /content/drive
49a8bfe feat(ml): promotion gate v2 on all datasets; exact thresholds; hardened Colab run
175ce04 fix(ml): Colab summary cell handles --only-lofo reports
3dd963e docs(ml): promotion gate 3.1-G.0 on unseen datasets; 3-fold results


In [3]:
# ===== Python 3.12 venv แยกจาก Colab (เวอร์ชันตรงกับที่ใช้เทรน candidate เดิม) =====
run("command -v uv >/dev/null || pip -q install uv; uv --version")
if not os.path.exists(VENV_PY):  # รันซ้ำใน runtime เดิม: uv venv จะถาม "replace it? [y/n]" แล้วค้างรอ input
    run("uv venv -q --python 3.12 /content/venv")
run(f"uv pip install -q --python {VENV_PY} -r {REPO_DIR}/ml/requirements-train.txt")
run(f"{VENV_PY} -c 'import sys, lightgbm, sklearn, libinjection; print(sys.version.split()[0], lightgbm.__version__, sklearn.__version__)'")

uv 0.12.15 (x86_64-unknown-linux-gnu)
3.12.3 4.7.0 1.9.0


0

In [4]:
# ===== เก็บ dataset / cache ไว้ใน Drive (symlink) =====
import shutil
def persist(rel):
    local, remote = f"{REPO_DIR}/{rel}", f"{DRIVE_DIR}/{rel.replace('/', '__')}"
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local):
        shutil.copytree(local, remote, dirs_exist_ok=True)
        shutil.rmtree(local)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    os.symlink(remote, local)

if USE_DRIVE:
    for rel in ("ml/dataset/external", "ml/dataset/.build_cache"):
        persist(rel)
    csic = f"{DRIVE_DIR}/csic_final.csv"
    if os.path.exists(csic) and not os.path.exists(f"{REPO_DIR}/ml/dataset/csic_final.csv"):
        shutil.copy2(csic, f"{REPO_DIR}/ml/dataset/")
run(f"ls -la {REPO_DIR}/ml/dataset")

total 26744
drwxr-xr-x 2 root root     4096 Sep 28 13:54 .
drwxr-xr-x 6 root root     4096 Sep 28 13:54 ..
lrwxrwxrwx 1 root root       55 Sep 28 13:54 .build_cache -> /content/drive/MyDrive/waf_ml/ml__dataset__.build_cache
-rw------- 1 root root 27376223 Sep 28 09:44 csic_final.csv
lrwxrwxrwx 1 root root       51 Sep 28 13:54 external -> /content/drive/MyDrive/waf_ml/ml__dataset__external


0

In [5]:
# ===== ข้อมูล VPS แบบ private (ถ้ามี) =====
tarballs = sorted(glob.glob(PRIVATE_TARBALL_GLOB)) if USE_DRIVE else []
if tarballs:
    print("แตกไฟล์:", tarballs[-1])
    run(f'tar -xzf "{tarballs[-1]}" -C {REPO_DIR}')
else:
    print("⚠️ ไม่มีข้อมูล VPS — รันด้วย public data อย่างเดียว (gate CORE = CSIC เท่านั้น)")
for rel in ("ml/dataset/vps_audit_labelled.jsonl", "ml/dataset/nginx_real_benign.jsonl", "ml/telemetry"):
    print("✅" if os.path.exists(f"{REPO_DIR}/{rel}") else "—", rel)

⚠️ ไม่มีข้อมูล VPS — รันด้วย public data อย่างเดียว (gate CORE = CSIC เท่านั้น)
— ml/dataset/vps_audit_labelled.jsonl
— ml/dataset/nginx_real_benign.jsonl
— ml/telemetry


In [6]:
# ===== Public dataset: open-appsec + SR-BH 2020 (ตรวจ sha256) — ข้ามถ้าเคยเตรียมไว้แล้ว =====
if EXPERIMENT_MODE != "gate" or RUN_TRAINER or RUN_UNIT_EXPERIMENT:
    if not os.path.exists("ml/dataset/external/prepare_stats.json"):
        run(f"PYTHONPATH=. {VENV_PY} ml/prepare_external_datasets.py --download", log="colab_logs/prepare_external.log")
    run("ls -la ml/dataset/external")
    # หยุดที่นี่ถ้าเตรียมไม่สำเร็จ — ถ้ารันต่อจะได้ผลจาก CSIC อย่างเดียวซึ่งใช้ตัดสินเรื่องฟีเจอร์ไม่ได้
    assert os.path.exists("ml/dataset/external/prepare_stats.json"), "เตรียม public dataset ไม่สำเร็จ — ดู colab_logs/prepare_external.log"
    if USE_DRIVE and os.path.exists("ml/dataset/csic_final.csv") and not os.path.exists(f"{DRIVE_DIR}/csic_final.csv"):
        shutil.copy2("ml/dataset/csic_final.csv", f"{DRIVE_DIR}/")

lrwxrwxrwx 1 root root 51 Sep 28 13:54 ml/dataset/external -> /content/drive/MyDrive/waf_ml/ml__dataset__external


In [7]:
# ===== Unit tests =====
# test ล้ม = หยุด Run all (ผลที่ได้จากโค้ดที่ test ไม่ผ่านใช้ไม่ได้)
run(f"PYTHONPATH=. {VENV_PY} -m pytest ml/test_value_features.py ml/test_feature_engineering.py "
    f"ml/test_promotion_gate.py ml/tests -q -p no:warnings", log="colab_logs/unit_tests.log")

..........................................                               [100%]
42 passed in 5.52s


0

In [8]:
# ===== 1. ทดลอง value features (config ค่าตั้งต้นใน ml/experiment_value_features.py: A, B, D, E, F = E_plus_query_body_entropy, D+qbe) =====
if EXPERIMENT_MODE in ("full", "lofo"):
    flag = "--only-lofo" if EXPERIMENT_MODE == "lofo" else ""
    run(f"PYTHONPATH=. {VENV_PY} -W ignore ml/experiment_value_features.py {flag}", log="colab_logs/value_features.log")

[+] CSIC Dataset already exists at: /content/waf_project/ml/dataset/csic_final.csv
[+] Loaded built dataset from cache dataset/.build_cache/a8fb203f8addfaab845ad7fa.joblib
[+] Loaded value features from cache dataset/.build_cache/value_56268575ffb8b6e7c5d8dddd.joblib
[*] LightGBM params from task3-1-real-augmented-candidate-20260927-205652; sources present: ['CSIC_2010_Cleaned', 'OpenAppSec_Legitimate', 'OpenAppSec_Malicious', 'SRBH2020_Honeypot']
[*] Attack rows with any value-detector hit: 76.8% (benign 0.65%)

[*] Fold 0: building context-transfer stress set (evaluation only)
[*] Stress set: 1500 injected + 1500 control (payload pool 13096, context pool 7156)

[*] Fold 1: building context-transfer stress set (evaluation only)
[*] Stress set: 1500 injected + 1500 control (payload pool 13066, context pool 7278)

[*] Fold 2: building context-transfer stress set (evaluation only)
[*] Stress set: 1500 injected + 1500 control (payload pool 13086, context pool 7275)

 A_base36: 36 features

In [9]:
# ===== 2. เทรน hybrid candidate (hyperparameter grid + holdout) =====
if RUN_TRAINER:
    run(f"PYTHONPATH=. {VENV_PY} -W ignore ml/train_gen3_full_real_benchmark.py", log="colab_logs/train_gen3.log")
    run(f"PYTHONPATH=. {VENV_PY} -W ignore ml/test_comprehensive.py", log="colab_logs/scenarios.log")

In [10]:
# ===== 3. โมเดลราย unit (MIL) + context-transfer stress test + LOSO =====
if RUN_UNIT_EXPERIMENT:
    run(f"PYTHONPATH=. {VENV_PY} -W ignore ml/experiment_unit_model.py", log="colab_logs/unit_model.log")

In [11]:
# ===== สรุปผลของรอบนี้ + Promotion Gate 3.1-G.0 =====
import json, pandas as pd
fmt = lambda d: f"{d['mean']*100:.1f} ± {d['std']*100:.1f}" if d else None
archive = "ml/models/archive/experiment-value-features-*/value_feature_report.json"
mine = sorted((p for p in glob.glob(archive) if os.path.getmtime(p) >= SESSION_START), key=os.path.getmtime)
if EXPERIMENT_MODE in ("full", "lofo"):
    assert mine, "ไม่พบรายงานของรอบนี้ — การทดลองไม่ได้บันทึกผล (ดู colab_logs/value_features.log)"
for path in mine:  # เฉพาะรายงานที่สร้างในรอบนี้ ไม่หยิบรายงานเก่ามาแสดง
    rep = json.load(open(path))
    rows, lofo = [], []
    for name, e in rep["configs"].items():
        s = e.get("summary") or {}  # ว่างเมื่อรันแบบ lofo
        if s:
            rows.append({"config": name, "features": e["n_features"], "CSIC benign": fmt(s["core_benign"]),
                         "CSIC attack": fmt(s["core_attack"]), "LFI": fmt(s["lfi"]), "SQLi": fmt(s["sqli"]),
                         "stress detect": fmt(s["stress_detect"]), "stress FP": fmt(s["stress_control_fp"]),
                         "scenario attack": s["scenario_attack_blocked"]["mean"], "scenario normal": s["scenario_normal_allowed"]["mean"]})
        for held, r in e.get("leave_one_dataset_out", {}).items():
            t = r["at_training_threshold"]
            lofo.append({"config": name, "held out": held, "AUC": r.get("auc"),
                         "attack @ benign 98.5%": r.get("attack_recall_at_benign_98_5"),
                         "attack @ benign 99.9%": r.get("attack_recall_at_benign_99_9"),
                         "training-thr benign": t["benign_recall"], "training-thr attack": t["attack_recall"]})
    print(path, "| sources:", rep["sources"], "| holdout folds:", rep.get("holdout_folds"))
    if rows:
        display(pd.DataFrame(rows).set_index("config"))
    if lofo:
        display(pd.DataFrame(lofo).set_index(["held out", "config"]).sort_index())

# Gate: รวมรายงานในเครื่องกับที่เก็บใน Drive — holdout fold จากรายงานล่าสุดที่มี fold,
# leave-one-dataset-out จากรายงานล่าสุดที่ใช้ตัววัดแบบ exact (ดู ml/promotion_gate.py)
pool = glob.glob(archive)
if USE_DRIVE:
    pool += glob.glob(f"{DRIVE_DIR}/results/*/experiment-value-features-*/value_feature_report.json")
if pool:
    reports = " ".join(f'"{p}"' for p in pool)
    run(f"PYTHONPATH=. {VENV_PY} ml/promotion_gate.py {reports} --json colab_logs/gate.json", log="colab_logs/gate.log")
else:
    print("ยังไม่มีรายงานให้ประเมิน gate")

ml/models/archive/experiment-value-features-20260928-144424/value_feature_report.json | sources: ['CSIC_2010_Cleaned', 'OpenAppSec_Legitimate', 'OpenAppSec_Malicious', 'SRBH2020_Honeypot'] | holdout folds: 3


,features,CSIC benign,CSIC attack,LFI,SQLi,stress detect,stress FP,scenario attack,scenario normal
config,,,,,,,,,
A_base36,36,98.7 ± 0.6,58.5 ± 1.1,99.9 ± 0.0,91.4 ± 0.5,27.0 ± 0.7,0.1 ± 0.0,19.6667,37.0
B_36_plus_value,53,98.5 ± 0.5,60.3 ± 0.5,99.9 ± 0.1,95.4 ± 0.2,42.4 ± 1.4,0.0 ± 0.0,23.0000,37.0
D_no_context_monotone,47,98.4 ± 0.5,51.6 ± 1.0,92.7 ± 1.1,92.9 ± 0.2,67.7 ± 1.8,0.4 ± 0.1,20.6667,37.0
E_no_context,47,98.4 ± 0.2,56.9 ± 0.7,93.5 ± 0.9,95.1 ± 0.3,57.9 ± 2.1,0.2 ± 0.1,21.3333,37.0
E_plus_query_body_entropy,48,98.7 ± 0.1,56.6 ± 0.5,95.0 ± 0.9,95.0 ± 0.1,60.1 ± 1.0,0.1 ± 0.1,23.0000,37.0
D_plus_query_body_entropy,48,98.4 ± 0.5,53.2 ± 0.8,94.2 ± 0.8,93.3 ± 0.2,61.1 ± 2.8,0.2 ± 0.0,21.6667,37.0


AUC  attack @ benign 98.5%  \
held out          config                                                     
CSIC_2010_Cleaned A_base36                   0.6997                 0.3081   
                  B_36_plus_value            0.7142                 0.3868   
                  D_no_context_monotone      0.7054                 0.3956   
                  D_plus_query_body_entropy  0.7092                 0.3975   
                  E_no_context               0.6852                 0.2453   
                  E_plus_query_body_entropy  0.7052                 0.4096   
OpenAppSec        A_base36                   0.7766                 0.0953   
                  B_36_plus_value            0.9372                 0.8077   
                  D_no_context_monotone      0.9381                 0.6605   
                  D_plus_query_body_entropy  0.9412                 0.7596   
                  E_no_context               0.9444                 0.7594   
                  E_plus_query_body_entropy  0.9510                 0.7803   
SRBH2020_Honeypot A_base36                   0.7267                 0.0525   
                  B_36_plus_value            0.9067                 0.3880   
                  D_no_context_monotone      0.8625                 0.1709   
                  D_plus_query_body_entropy  0.9064                 0.2750   
                  E_no_context               0.8854                 0.2262   
                  E_plus_query_body_entropy  0.9247                 0.6429   

                                             attack @ benign 99.9%  \
held out          config                                             
CSIC_2010_Cleaned A_base36                                  0.2564   
                  B_36_plus_value                           0.2997   
                  D_no_context_monotone                     0.2379   
                  D_plus_query_body_entropy                 0.2286   
                  E_no_context                              0.2420   
                  E_plus_query_body_entropy                 0.2263   
OpenAppSec        A_base36                                  0.0068   
                  B_36_plus_value                           0.0460   
                  D_no_context_monotone                     0.0341   
                  D_plus_query_body_entropy                 0.0459   
                  E_no_context                              0.0511   
                  E_plus_query_body_entropy                 0.0517   
SRBH2020_Honeypot A_base36                                  0.0094   
                  B_36_plus_value                           0.0047   
                  D_no_context_monotone                     0.1140   
                  D_plus_query_body_entropy                 0.1476   
                  E_no_context                              0.1028   
                  E_plus_query_body_entropy                 0.2866   

                                             training-thr benign  \
held out          config                                           
CSIC_2010_Cleaned A_base36                                0.9993   
                  B_36_plus_value                         0.9992   
                  D_no_context_monotone                   0.9978   
                  D_plus_query_body_entropy               0.9977   
                  E_no_context                            0.9606   
                  E_plus_query_body_entropy               0.9983   
OpenAppSec        A_base36                                0.3489   
                  B_36_plus_value                         0.3936   
                  D_no_context_monotone                   0.6559   
                  D_plus_query_body_entropy               0.5497   
                  E_no_context                            0.6242   
                  E_plus_query_body_entropy               0.3720   
SRBH2020_Honeypot A_base36                                0.9948   
                  B_36_plus_value                         0.9955  

== A_base36: FAILED (3 holdout folds)
  [PASS] G1_benign_every_dataset                                      0.9866  target >= 0.985  | CSIC_2010_Cleaned=0.9866, OpenAppSec=0.9997, SRBH2020_Honeypot=0.9983
  [FAIL] G2_known_attack_mean                                           0.84  target >= 0.85  | CSIC_2010_Cleaned=0.585, OpenAppSec=0.9996, SRBH2020_Honeypot=0.9355
  [FAIL] G3_unseen_attack_mean                                         0.152  target >= 0.7  | CSIC_2010_Cleaned=0.3081, OpenAppSec=0.0953, SRBH2020_Honeypot=0.0525
  [FAIL] G4_no_dataset_below                                          0.0525  target >= 0.4  | unseen:SRBH2020_Honeypot=0.0525
  [FAIL] G5_sanity                  {'scenario_normal_min': 37, 'scenario_attack_mean': 19.67, 'stress_detect_mean': 0.27}  target normal 37/37, attack >= 23/26, stress >= 0.6  | stress_control_fp_mean=0.0005
     from: {'holdout_folds': 'ml/models/archive/experiment-value-features-20260928-144424/value_feature_report.json', 'leave_one_

In [12]:
# ===== รวมผลเป็น zip (รายงาน JSON + log; ไม่รวมไฟล์โมเดล .joblib ซึ่งใหญ่) =====
stamp = time.strftime("%Y%m%d-%H%M%S")
new_dirs = [d for d in glob.glob("ml/models/archive/*") if os.path.isdir(d) and os.path.getmtime(d) >= SESSION_START]
print("ผลลัพธ์รอบนี้:", new_dirs)
out = f"/content/waf_colab_results_{stamp}.zip"
targets = " ".join(new_dirs + ["colab_logs"])
run(f'zip -qr {out} {targets} -x "*.joblib"')
if USE_DRIVE:
    os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
    shutil.copy2(out, f"{DRIVE_DIR}/results/")
    # รายงาน JSON + log แบบไม่บีบอัด ให้ Claude อ่านผ่าน Google Drive ได้โดยตรง
    run_dir = f"{DRIVE_DIR}/results/{stamp}"
    for d in new_dirs + ["colab_logs"]:
        shutil.copytree(d, f"{run_dir}/{os.path.basename(d)}", ignore=shutil.ignore_patterns("*.joblib"), dirs_exist_ok=True)
    print("บันทึกผลไว้ที่", run_dir)
    # โมเดลเก็บไว้ใน Drive (ไม่ขึ้น git)
    for d in new_dirs:
        for f in glob.glob(f"{d}/*.joblib"):
            os.makedirs(f"{DRIVE_DIR}/models/{os.path.basename(d)}", exist_ok=True)
            shutil.copy2(f, f"{DRIVE_DIR}/models/{os.path.basename(d)}/")
try:
    from google.colab import files
    files.download(out)
except Exception as exc:  # ผลอยู่ใน Drive แล้ว ไม่ต้องดาวน์โหลดก็ได้
    print("ดาวน์โหลดอัตโนมัติไม่สำเร็จ:", exc)

ผลลัพธ์รอบนี้: ['ml/models/archive/task3-1-real-augmented-candidate-20260925-120054', 'ml/models/archive/task3-1-real-augmented-candidate-20260924-152133', 'ml/models/archive/task2-1-real-fn-audit-20260913', 'ml/models/archive/task3-1-real-augmented-20260924-145933', 'ml/models/archive/task3-1-real-augmented-candidate-20260927-165254', 'ml/models/archive/task3-1-real-augmented-candidate-20260924-164455', 'ml/models/archive/task3-1-real-augmented-candidate-20260927-205652', 'ml/models/archive/pre-gen32-baseline-20260913', 'ml/models/archive/task2-2-cost-sensitive-20260913', 'ml/models/archive/task3-1-real-augmented-candidate-20260924-164632', 'ml/models/archive/experiment-value-features-20260928-144424', 'ml/models/archive/task2-3-model-exploration-20260924-142211', 'ml/models/archive/task3-1-real-augmented-candidate-20260927-204037', 'ml/models/archive/experiment-content-model-20260927-205001', 'ml/models/archive/task2-1-audited-training-candidate-20260913', 'ml/models/archive/task2-3-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## เอาผลกลับไปวิเคราะห์

ผลอยู่ใน Google Drive ที่ `MyDrive/waf_ml/results/<วันเวลา>/` (และ log ระหว่างรันที่ `MyDrive/waf_ml/live_logs/`, ผล gate ที่ `live_logs/gate.log`) — บอก Claude ใน VS Code ว่า *"วิเคราะห์ผล value features ล่าสุด"* ได้เลย ไม่ต้องดาวน์โหลดเอง

การอ่านผล:
1. **Promotion Gate 3.1-G.0** (ตารางจาก `ml/promotion_gate.py`) — ตัดสินว่า candidate พร้อมไปขั้นถัดไปหรือยัง
2. **leave-one-dataset-out** — AUC และ attack @ benign 98.5% บน dataset ที่โมเดลไม่เคยเห็น = การป้องกันเว็บลูกค้าใหม่ (threshold จูนราย tenant ตามขั้น Silent → Tuning)
3. **training-thr benign** ต่ำ = ถ้าใช้ threshold ตายตัวกับเว็บใหม่จะบล็อกคนปกติ → ต้องผ่าน Silent → Tuning ก่อน enforce เสมอ
4. ค่า ± คือความแกว่งระหว่าง holdout fold — ความต่างที่เล็กกว่านี้ถือว่าแยกไม่ออก